In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl

In [ ]:
# Load the model checkpoint
checkpoint_path = 'vae45_relu_Br_beta25_lt006_bz064_splt80_ep1000_lr5e-4.pth'
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Save the extracted weights to a new file
output_path = 'vae_pre_trained_w.pt'
torch.save(checkpoint.state_dict(), output_path)

print(f"Model weights extracted and saved to {output_path}")

Model weights extracted and saved to vae_pre_trained_w.pt


In [ ]:
vae_dict = torch.load("vae_pre_trained_w.pt", weights_only=True)
print(vae_dict.keys())

# Remove keys starting with 'decoder' and 'encoder.0'
keys_to_remove = [key for key in vae_dict.keys() if key.startswith('decoder') or key.startswith('encoder.0')]
for key in keys_to_remove:
    del vae_dict[key]

# Rename keys encoder.1 -> encoder.0, encoder.2 -> encoder.1, encoder.3 -> encoder.2, ...
encoder_dict = {}
for key in vae_dict.keys():
    if key.startswith('encoder.'):
        parts = key.split('.')
        layer_num = int(parts[1]) - 1
        new_key = f'encoder.{layer_num}.' + '.'.join(parts[2:])
        encoder_dict[new_key] = vae_dict[key]
    else:
        encoder_dict[key] = vae_dict[key]

print(encoder_dict.keys())
torch.save(encoder_dict, "encoder_pre_trained_w.pt")

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split

img_path = 'dataset/'


# Dataset with Transformation
dataset = datasets.ImageFolder(
    img_path,
    transforms.Compose([
        transforms.Resize((128, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
              mean=[0.0, 0.0, 0.0], 
              std=[1.0, 1.0, 1.0])])
)



# Data split
trainpctg  = 0.8
train_size = int(trainpctg * len(dataset))
test_size = len(dataset) - train_size
trainset, testset = random_split(dataset, [train_size, test_size])

print(f"Training set size: {len(trainset)}, Testing set size: {len(testset)}")

In [ ]:
import torch

# Get the first image and its label from the dataset
image, label = dataset[0]

# Convert the image tensor to a byte tensor
image_bytes = torch.ByteTensor(image.mul(255).byte())

# Save the byte tensor to a binary file
with open('test_image.bin', 'wb') as f:
    f.write(image_bytes.numpy().tobytes())

print("First image exported to 'test_image.bin'")